In [20]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import date

PROJECT_ROOT = Path("..").resolve()
RAW_DIR  = PROJECT_ROOT / "data" / "raw"
DB_PATH  = PROJECT_ROOT / "data" / "hospital_typology.db"
# SNAPSHOT = date.today().strftime("%Y%m%d")  # comment out
SNAPSHOT = "20260520"  # HELLLOOOO use the actual date of the raw files - HERE NEEDS CHANGE 

DATASETS = [
    "hospital_general_info",
    "readmissions_and_deaths",
    "medicare_spending",
    "healthcare_infections",
    "hcahps_patient_survey",
    "unplanned_visits",
    "timely_effective_care",
]

con = sqlite3.connect(DB_PATH)

for name in DATASETS:
    csv_path = RAW_DIR / f"{name}_{SNAPSHOT}.csv"
    table    = f"stg_{name}"

    if not csv_path.exists():
        print(f"  {csv_path.name} not found — skipping")
        continue

    df = pd.read_csv(csv_path, low_memory=False, dtype={"Facility ID": str})
    df.to_sql(table, con, if_exists="replace", index=False)
    print(f"{table:<38} {len(df):>7,} rows × {len(df.columns)} cols")




con.close()
print(f"\n Database: {DB_PATH}")


## def latest_snapshot(name: str) -> str | None:
##    """Find the most recent snapshot date for a given dataset."""
##    files = sorted(RAW_DIR.glob(f"{name}_*.csv"))
##    if not files:
##        return None
    # Filename: hospital_general_info_20260520.csv → 20260520
##    return files[-1].stem.split("_")[-1]

##for name in DATASETS:
##    snapshot = latest_snapshot(name)
##    if snapshot is None:
##        print(f"  No file found for {name} — skipping")
##        continue
    
##    csv_path = RAW_DIR / f"{name}_{snapshot}.csv"
##    table    = f"stg_{name}"
    
##    df = pd.read_csv(csv_path, low_memory=False, dtype={"Facility ID": str})
####    df.to_sql(table, con, if_exists="replace", index=False)
##    print(f" {table:<38} {len(df):>7,} rows × {len(df.columns)} cols  ({snapshot})")

stg_hospital_general_info                5,432 rows × 38 cols
stg_readmissions_and_deaths             95,840 rows × 18 cols
stg_medicare_spending                   63,646 rows × 13 cols
stg_healthcare_infections              172,512 rows × 15 cols
stg_hcahps_patient_survey              325,856 rows × 22 cols
stg_unplanned_visits                    67,088 rows × 20 cols
stg_timely_effective_care              138,173 rows × 16 cols

 Database: C:\Users\Fatima zahra\Projects\CMS-hospital-typology\data\hospital_typology.db


In [31]:
import sqlite3
import pandas as pd
from pathlib import Path

# Re-establish environment paths so the cell is self-contained
PROJECT_ROOT = Path("..").resolve()
DB_PATH   = PROJECT_ROOT / "data" / "hospital_typology.db"
ipps_path = PROJECT_ROOT / "data" / "raw" / "fy_2026_ipps_final_rule_impact_file" / "FY 2026 IPPS Final Rule Impact File.txt"

# === Step 1: Read the IPPS Impact File (tab-delimited txt) ===
# skiprows=1 skips the title row above the real headers
df_ipps_raw = None
for encoding in ['utf-8', 'cp1252', 'latin-1']:
    try:
        df_ipps_raw = pd.read_csv(ipps_path, sep='\t', encoding=encoding, skiprows=1)
        print(f"Loaded with encoding='{encoding}': "
              f"{df_ipps_raw.shape[0]} rows × {df_ipps_raw.shape[1]} cols")
        break
    except UnicodeDecodeError:
        continue

if df_ipps_raw is None:
    raise ValueError("Couldn't decode the file with any common encoding")

# === Step 2: Keep only the 6 columns we need ===
cols_to_keep = {
    'Provider Number':       'ccn',                    # 6-digit CCN
    'Beds':                  'bed_count',              # total beds (cost report data)
    'TACMIV43':              'cmi',                    # transfer-adjusted CMI, V43 grouper
    'Resident to Bed Ratio': 'resident_to_bed_ratio',  # 0 = non-teaching, >0 = teaching
    'DSHPCT':                'dsh_pct',                # safety-net burden (Medicaid + SSI ratio)
    'URGEO':                 'urban_rural_geo',        # 'U' or 'R' coarse fallback
}

missing = [c for c in cols_to_keep if c not in df_ipps_raw.columns]
if missing:
    print(f"\nColumns not found: {missing}")
    print(f"All columns in file: {df_ipps_raw.columns.tolist()}")
    raise KeyError("Adjust the names in cols_to_keep to match what's in the file")

stg_ipps = df_ipps_raw[list(cols_to_keep)].rename(columns=cols_to_keep)
print(f"\nFiltered to {stg_ipps.shape[1]} columns, {stg_ipps.shape[0]} rows:")
print(stg_ipps.head())

# === Step 3: Open a fresh connection and save to SQLite ===
con = sqlite3.connect(DB_PATH)
try:
    stg_ipps.to_sql('stg_ipps', con, if_exists='replace', index=False)
    print(f"\nSaved to SQLite as stg_ipps ({stg_ipps.shape[0]} rows)")
finally:
    con.close()

print(f"Database closed cleanly. Path: {DB_PATH}")

Loaded with encoding='cp1252': 3103 rows × 62 cols

Filtered to 6 columns, 3103 rows:
     ccn  bed_count     cmi  resident_to_bed_ratio  dsh_pct urban_rural_geo
0  10001        345  1.9757                 0.1241   0.2752          OURBAN
1  10005        182  1.5027                 0.0000   0.3164           RURAL
2  10006        223  1.7048                 0.2451   0.2040          OURBAN
3  10007         45  1.2341                 0.0000   0.2580           RURAL
4  10011        284  1.8662                 0.0948   0.2157          LURBAN

Saved to SQLite as stg_ipps (3103 rows)
Database closed cleanly. Path: C:\Users\Fatima zahra\Projects\CMS-hospital-typology\data\hospital_typology.db
